In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, inspect
from IPython.display import display
import json

# 1. COMANDOS PARA RECARGA AUTOMÁTICA
%load_ext autoreload
%autoreload 2
%matplotlib inline

pd.set_option('display.max_columns', None)   # mostrar todas las columnas
pd.set_option('display.width', 0)           # dejar que use todo el ancho disponible
pd.set_option('display.max_colwidth', None) # Quitar el límite de ancho de las columnas
pd.set_option('display.expand_frame_repr', False) # Para que no "envuelva" la tabla y se mantenga en una sola fila larga

# Descarga de bases de datos:

In [ ]:
def load_all_databases_to_dict(data_dir="../.data"):
    """
    Carga todas las bases de datos SQLite (.db) en el directorio indicado.
    Retorna un diccionario: {db_name: {table_name: DataFrame}}
    Reconstruye tablas heredadas (como efficiency_department y tactical_department) 
    a partir del nuevo esquema unificado (unified_department y analysis_layer).
    Procesa también las columnas con formatos JSON.
    """
    import os
    from pathlib import Path
    from sqlalchemy import create_engine, inspect
    import pandas as pd
    import json

    path = Path(data_dir)
    if not path.exists():
        print(f"❌ El directorio '{data_dir}' no existe.")
        return {}

    db_files = sorted(list(path.glob("*.db")))
    if not db_files:
        print(f"❌ No se encontraron archivos .db en '{data_dir}'.")
        return {}

    all_dfs = {}
    json_cols_to_parse = [
        'emotions', 'behavioral_errors', 'cognitive_patterns', 
        'pre_trade_emotions', 'mid_trade_emotions', 'post_trade_emotions', 
        'confirmation_params'
    ]

    for db_file in db_files:
        db_name = db_file.stem
        print(f"\n📂 Procesando base de datos: '{db_file.name}'...")
        
        try:
            engine = create_engine(f"sqlite:///{db_file}")
            inspector = inspect(engine)
            nombres_tablas = inspector.get_table_names()
        except Exception as e:
            print(f"   ❌ Error al conectar o inspeccionar '{db_file.name}': {e}")
            continue

        if not nombres_tablas:
            print(f"   ⚠️ No se encontraron tablas en '{db_file.name}'.")
            all_dfs[db_name] = {}
            continue

        db_tables = {}
        for tabla in nombres_tablas:
            print(f"   📊 Cargando tabla: '{tabla}'...")
            try:
                df = pd.read_sql(f"SELECT * FROM {tabla}", engine)
            except Exception as e:
                print(f"      ❌ Error al leer tabla '{tabla}': {e}")
                continue

            # Procesar columnas JSON si la tabla las tiene
            for col in json_cols_to_parse:
                if col in df.columns:
                    def parse_json(x):
                        if not x or pd.isna(x): return {}
                        try:
                            res = json.loads(x) if isinstance(x, str) else x
                            if isinstance(res, list):
                                return {f"item_{i}": v for i, v in enumerate(res)}
                            if not isinstance(res, dict):
                                return {"value": res}
                            return res
                        except (json.JSONDecodeError, TypeError):
                            return {}

                    # Aplanamos el JSON de esa columna específica
                    df_json = pd.json_normalize(df[col].apply(parse_json))
                    if not df_json.empty:
                        df_json.columns = [f"{col}_{subcol}" for subcol in df_json.columns]
                        df = pd.concat([df.drop(columns=[col]), df_json], axis=1)

            db_tables[tabla] = df
            print(f"      ✅ {tabla}: {df.shape[0]} filas, {df.shape[1]} columnas.")

        # Reconstrucción de tablas heredadas para mantener compatibilidad si no existen pero sí el esquema unificado
        if 'unified_department' in db_tables and 'analysis_layer' in db_tables:
            unified_df = db_tables['unified_department']
            analysis_df = db_tables['analysis_layer']

            if 'efficiency_department' not in db_tables and not unified_df.empty:
                print("   🔄 Reconstruyendo 'efficiency_department' desde esquema unificado...")
                try:
                    eff_layers = analysis_df[analysis_df['department'] == 'EFFICIENCY']
                    if not eff_layers.empty:
                        eff_pivot = eff_layers.pivot(index='trade_id', columns='layer_name', values=['direction', 'strength'])
                        eff_pivot.columns = [f"{layer.lower()}_{val}" for val, layer in eff_pivot.columns]
                        eff_pivot = eff_pivot.reset_index().rename(columns={'trade_id': 'id'})
                        eff_cols = [c for c in ['id', 'state', 'asset', 'created_at', 'updated_at', 'market_bias', 'calc_edge'] if c in unified_df.columns]
                        db_tables['efficiency_department'] = pd.merge(unified_df[eff_cols], eff_pivot, on='id', how='left')
                        print("      ✅ Reconstrucción exitosa.")
                    else:
                        db_tables['efficiency_department'] = pd.DataFrame()
                except Exception as e:
                    print(f"      ⚠️ No se pudo reconstruir 'efficiency_department': {e}")

            if 'tactical_department' not in db_tables and not unified_df.empty:
                print("   🔄 Reconstruyendo 'tactical_department' desde esquema unificado...")
                try:
                    tac_layers = analysis_df[analysis_df['department'] == 'TACTICAL']
                    if not tac_layers.empty:
                        tac_pivot = tac_layers.pivot(index='trade_id', columns='layer_name', values=['direction', 'strength', 'score', 'thesis'])
                        tac_pivot.columns = [f"{layer.lower()}_{val}" for val, layer in tac_pivot.columns]
                        tac_pivot = tac_pivot.reset_index().rename(columns={'trade_id': 'id'})
                        tac_cols = [c for c in ['id', 'p4_hierarchy', 'p1_timeframe', 'p1_type', 'nodes_l1', 'nodes_l2', 'tactical_classification', 'calc_edge', 'long_prob', 'short_prob', 'no_trade_prob'] if c in unified_df.columns]
                        db_tables['tactical_department'] = pd.merge(unified_df[tac_cols], tac_pivot, on='id', how='left')
                        print("      ✅ Reconstrucción exitosa.")
                    else:
                        db_tables['tactical_department'] = pd.DataFrame()
                except Exception as e:
                    print(f"      ⚠️ No se pudo reconstruir 'tactical_department': {e}")

        all_dfs[db_name] = db_tables

    return all_dfs

In [ ]:
# --- EJECUCIÓN ---
all_dfs = load_all_databases_to_dict()

# Mostramos bases de datos cargadas
print("\nBases de datos disponibles:", list(all_dfs.keys()))

# Para mantener compatibilidad con las celdas heredadas de abajo, seleccionamos una base de datos activa.
# Priorizamos 'flight_account_001_xauusd' por contener datos, o la primera que no esté vacía.
active_db = 'flight_account_001_xauusd'
if active_db not in all_dfs or not all_dfs[active_db]:
    # Buscar una que tenga tablas
    non_empty_dbs = [name for name, tables in all_dfs.items() if tables]
    active_db = non_empty_dbs[0] if non_empty_dbs else (list(all_dfs.keys())[0] if all_dfs else None)

if active_db:
    print(f"👉 Seleccionando '{active_db}' como base de datos activa para 'dfs'.")
    dfs = all_dfs[active_db]
else:
    print("❌ No se encontró ninguna base de datos activa.")
    dfs = {}

In [ ]:
dfs['efficiency_audit']

In [ ]:
dfs['efficiency_department']

In [ ]:
dfs['tactical_department']

In [ ]:
dfs['tactical_audit']